# STAGE 1 - IMPORTS AND CONFIGURATION

In [1]:
import os
import time
import asyncio
import logging
import numpy as np
import pandas as pd
import nltk

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
# Ollama integrations
from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate

from bert_score import BERTScorer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

from deepeval.test_case import LLMTestCase, LLMTestCaseParams

from deepeval.metrics import (
    GEval,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric
)

from deepeval.models import OllamaModel


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL = "llama3.1"

# Local Ollama server
OLLAMA_BASE_URL = "http://localhost:11434"

# Local embeddings
EMBEDDING_MODEL = "nomic-embed-text"


# ============================================================
# 2. FILE CONFIGURATION
# ============================================================

INPUT_CSV = "ai_teacher_20_questions.csv"
PDF_FILE = "HopeAI_Teaching.pdf"
OUTPUT_CSV = "RAG_LLM_FINAL_EVALUATION_OLLAMA.csv"


# ============================================================
# 3. LOGGING
# ============================================================

logging.getLogger("deepeval").setLevel(logging.CRITICAL)
logging.getLogger("transformers").setLevel(logging.ERROR)


# ============================================================
# 4. NLTK
# ============================================================

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


# ============================================================
# 5. LOAD DATA
# ============================================================

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "User Query",
    "Teacher Response"
]

missing = set(required_columns) - set(df.columns)

if missing:
    raise ValueError(
        f"Missing CSV columns: {missing}"
    )

df = df.dropna(
    subset=required_columns
).reset_index(drop=True)

df["User Query"] = (
    df["User Query"]
    .astype(str)
    .str.strip()
)

df["Teacher Response"] = (
    df["Teacher Response"]
    .astype(str)
    .str.strip()
)

print(f"Questions loaded: {len(df)}")


# ============================================================
# 6. OLLAMA LLM
# ============================================================

llm = ChatOllama(
    model=MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0
)

print("Ollama Llama 3.1 loaded!")


# ============================================================
# 7. OLLAMA EMBEDDINGS
# ============================================================

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL
)

print("Ollama embedding model loaded!")

C:\Users\manju\AppData\Local\Temp\ipykernel_3412\782346669.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\manju\AppData\Local\Temp\ipykernel_3412\782346669.py:21: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Questions loaded: 20
Ollama Llama 3.1 loaded!
Ollama embedding model loaded!


# STAGE 2 - RAG PIPELINE

In [2]:
PDF_FILE = "AI_Teaching.pdf"

# Load PDF
documents = PyPDFLoader(PDF_FILE).load()

# Chunk documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

# Ollama embeddings
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

# FAISS
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Ollama LLM
llm = ChatOllama(
    model="llama3.1",
    base_url="http://localhost:11434",
    temperature=0
)

print("PDF pages loaded:", len(documents))
print("Chunks created:", len(chunks))
print("FAISS vector store ready.")
print("Ollama LLM ready.")

PDF pages loaded: 6
Chunks created: 14
FAISS vector store ready.
Ollama LLM ready.


# STAGE 3 - DEFINE EVALUATION METRICS

In [3]:
# ============================================================
# SIMILARITY METRICS
# ============================================================

bert_scorer = BERTScorer(
    lang="en",
    rescale_with_baseline=False
)

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)


# ============================================================
# OLLAMA EVALUATION MODEL
# ============================================================

EVALUATION_MODEL = "llama3.1"

judge_model = OllamaModel(
    model=EVALUATION_MODEL,
    base_url="http://localhost:11434",
    temperature=0
)


# ============================================================
# ANSWER CORRECTNESS
# ============================================================

correctness_metric = GEval(
    name="Answer Correctness",
    model=judge_model,
    criteria="""
    Compare the generated answer with the reference answer.

    Evaluate:
    - factual accuracy
    - important information
    - contradictions
    - completeness

    Ignore minor wording differences.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)


# ============================================================
# HELPFULNESS
# ============================================================

helpfulness_metric = GEval(
    name="Helpfulness",
    model=judge_model,
    criteria="""
    Evaluate whether the answer:
    - directly addresses the question
    - is easy for a beginner to understand
    - provides a clear explanation
    - is useful to the learner
    - avoids unnecessary information
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    async_mode=False
)


# ============================================================
# ANSWER RELEVANCY
# ============================================================

answer_relevancy_metric = AnswerRelevancyMetric(
    model=judge_model,
    include_reason=False,
    async_mode=False
)


# ============================================================
# RAG GEVAL
# ============================================================

rag_geval_metric = GEval(
    name="RAG Correctness",
    model=judge_model,
    criteria="""
    Evaluate whether the generated answer is:

    1. Factually correct
    2. Consistent with the reference answer
    3. Complete enough to answer the question
    4. Free from important contradictions

    Ignore minor differences in wording.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)


# ============================================================
# RETRIEVAL / RAG METRICS
# ============================================================

context_precision_metric = ContextualPrecisionMetric(
    model=judge_model
)

context_recall_metric = ContextualRecallMetric(
    model=judge_model
)

faithfulness_metric = FaithfulnessMetric(
    model=judge_model
)

rag_answer_relevancy_metric = AnswerRelevancyMetric(
    model=judge_model,
    include_reason=False,
    async_mode=False
)


print("All Ollama evaluation metrics initialized.")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

All Ollama evaluation metrics initialized.


# STAGE 4 - INPUT QUERIES, GENERATE ANSWERS AND CALCULATE OPERATIONAL METRICS

In [4]:
# ============================================================
# DEFINE RETRIEVAL FUNCTION
# ============================================================

def get_context(query):
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    context = "\n\n".join(retrieval_context)

    return context, retrieval_context


print("SUCCESS: get_context() is defined.")

SUCCESS: get_context() is defined.


In [5]:
# ============================================================
# TEST get_context()
# ============================================================

test_query = "What is AI?"

context, retrieval_context = get_context(test_query)

print("Number of retrieved documents:", len(retrieval_context))
print("\nContext length:", len(context))

for i, ctx in enumerate(retrieval_context, 1):
    print(f"\n--- Retrieved Context {i} ---")
    print(ctx[:500])

Number of retrieved documents: 3

Context length: 2831

--- Retrieved Context 1 ---
AI Teaching: Principles, Methods, and Classroom
 Practice
A practical reference document for an AI Teacher and Retrieval-Augmented Generation (RAG) evaluation
project.
1. Introduction to AI in Education
What is Artificial Intelligence?
Artificial Intelligence (AI) refers to computer systems that can perform tasks that normally require human
intelligence. These tasks include understanding language, recognizing patterns, generating text, solving
problems, making predictions, and assisting with dec

--- Retrieved Context 2 ---
The AI teacher should explain RAG using the retrieved material. If the student asks for information that is not
present in the document, the AI should clearly state that the information could not be found in the provided
document rather than inventing an answer.
13. Key Terms
Artificial Intelligence
Computer systems designed to perform tasks associated with human intelligence.
Genera

In [6]:
from langchain_core.prompts import ChatPromptTemplate

# ============================================================
# RAG PROMPT
# ============================================================

rag_prompt = ChatPromptTemplate.from_template("""
You are an AI teacher.

Answer the student's question using only the information
provided in the retrieved context.

Rules:
- Give a clear and accurate explanation.
- Keep the explanation appropriate for a learner.
- Do not invent information that is not supported by the context.
- If the answer cannot be found in the context, say:
  "The information could not be found in the provided document."

Retrieved Context:
{context}

Student Question:
{question}

Answer:
""")

print("SUCCESS: rag_prompt is defined.")

SUCCESS: rag_prompt is defined.


In [7]:
# ============================================================
# TEST RAG GENERATION
# ============================================================

test_query = "What is AI?"

context, retrieval_context = get_context(test_query)

prompt_value = rag_prompt.invoke({
    "context": context,
    "question": test_query
})

response = llm.invoke(prompt_value)

print("Question:")
print(test_query)

print("\nGenerated Answer:")
print(response.content)

print("\nResponse Metadata:")
print(response.response_metadata)

Question:
What is AI?

Generated Answer:
According to the retrieved context, Artificial Intelligence (AI) is defined as:

"Artificial Intelligence (AI) refers to computer systems that can perform tasks that normally require human intelligence. These tasks include understanding language, recognizing patterns, generating text, solving problems, making predictions, and assisting with decisions."

So, AI is essentially computer systems that can perform tasks that are typically done by humans, using their intelligence.

Response Metadata:
{'model': 'llama3.1', 'created_at': '2026-10-08T18:58:45.0379833Z', 'done': True, 'done_reason': 'stop', 'total_duration': 42332506500, 'load_duration': 50331000, 'prompt_eval_count': 626, 'prompt_eval_duration': 25869629000, 'eval_count': 82, 'eval_duration': 16324477000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}


In [ ]:
from tqdm.auto import tqdm
import time
import numpy as np
import pandas as pd

results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="RAG generation"
):

    query = row["User Query"]
    reference = row["Teacher Response"]

    try:

        total_start = time.perf_counter()

        # ====================================================
        # RETRIEVAL
        # ====================================================

        start = time.perf_counter()

        context, retrieval_context = get_context(query)

        retrieval_latency = time.perf_counter() - start


        # ====================================================
        # GENERATION
        # ====================================================

        prompt_value = rag_prompt.invoke({
            "context": context,
            "question": query
        })

        start = time.perf_counter()

        response = llm.invoke(prompt_value)

        generation_latency = time.perf_counter() - start

        total_latency = time.perf_counter() - total_start

        generated = response.content.strip()


        # ====================================================
        # OLLAMA TOKEN USAGE
        # ====================================================

        metadata = getattr(
            response,
            "response_metadata",
            {}
        )

        input_tokens = metadata.get(
            "prompt_eval_count",
            0
        )

        output_tokens = metadata.get(
            "eval_count",
            0
        )

        total_tokens = (
            input_tokens + output_tokens
        )


        # ====================================================
        # OLLAMA TIMING
        # ====================================================

        ollama_total_duration = metadata.get(
            "total_duration",
            0
        )

        ollama_load_duration = metadata.get(
            "load_duration",
            0
        )

        ollama_prompt_eval_duration = metadata.get(
            "prompt_eval_duration",
            0
        )

        ollama_eval_duration = metadata.get(
            "eval_duration",
            0
        )


        # ====================================================
        # LOCAL COST
        # ====================================================

        input_cost = 0.0
        output_cost = 0.0
        total_cost = 0.0


        # ====================================================
        # SAVE RESULTS
        # ====================================================

        results.append({

            "User Query": query,

            "Teacher Response": reference,

            "GeneratedAnswer": generated,

            "RetrievedContext": retrieval_context,

            "RetrievalLatency": retrieval_latency,

            "GenerationLatency": generation_latency,

            "Latency": total_latency,

            "InputTokens": input_tokens,

            "OutputTokens": output_tokens,

            "TotalTokens": total_tokens,

            "OllamaTotalDuration": ollama_total_duration,

            "OllamaLoadDuration": ollama_load_duration,

            "OllamaPromptEvalDuration": ollama_prompt_eval_duration,

            "OllamaEvalDuration": ollama_eval_duration,

            "InputCost": input_cost,

            "OutputCost": output_cost,

            "TotalCost": total_cost

        })


    except Exception as e:

        print(
            f"\nError processing query: {query}\n"
            f"{type(e).__name__}: {e}"
        )

        results.append({

            "User Query": query,

            "Teacher Response": reference,

            "GeneratedAnswer": "",

            "RetrievedContext": [],

            "RetrievalLatency": np.nan,

            "GenerationLatency": np.nan,

            "Latency": np.nan,

            "InputTokens": np.nan,

            "OutputTokens": np.nan,

            "TotalTokens": np.nan,

            "OllamaTotalDuration": np.nan,

            "OllamaLoadDuration": np.nan,

            "OllamaPromptEvalDuration": np.nan,

            "OllamaEvalDuration": np.nan,

            "InputCost": np.nan,

            "OutputCost": np.nan,

            "TotalCost": np.nan

        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

successful = results_df["GeneratedAnswer"].ne("").sum()

print(f"\nCompleted: {successful}/{len(results_df)}")

display(results_df)

RAG generation:   0%|          | 0/20 [00:00<?, ?it/s]

# STAGE 5 - LLM EVALUATION - ANSWER QUALITY

In [15]:
quality_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="Answer quality"
):
    query = row["User Query"]
    generated = row["GeneratedAnswer"]
    reference = row["Teacher Response"]

    if not generated:
        quality_scores.append([np.nan] * 3)
        continue

    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference
    )

    scores = []

    for metric in [
        correctness_metric,
        answer_relevancy_metric,
        helpfulness_metric
    ]:
        try:
            metric.measure(test_case)
            scores.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            scores.append(np.nan)

    quality_scores.append(scores)


results_df[
    ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
] = pd.DataFrame(
    quality_scores,
    index=results_df.index
)

results_df["AnswerQuality"] = results_df[
    ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
].mean(axis=1)

display(results_df[
    [
        "User Query",
        "AnswerCorrectness",
        "AnswerRelevancy",
        "Helpfulness",
        "AnswerQuality"
    ]
])

Answer quality:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

Output()

# STAGE 6 - EVALUATE SIMILARITY METRICS

In [16]:
similarity_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="Similarity metrics"
):
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]

    if not generated:
        similarity_scores.append([np.nan] * 5)
        continue

    try:
        _, _, f1 = bert_scorer.score(
            [generated],
            [reference]
        )
        bert = float(f1[0])
    except Exception:
        bert = np.nan

    try:
        rouge_scores = rouge.score(reference, generated)

        rouge1 = rouge_scores["rouge1"].fmeasure
        rouge2 = rouge_scores["rouge2"].fmeasure
        rougeL = rouge_scores["rougeL"].fmeasure
    except Exception:
        rouge1 = rouge2 = rougeL = np.nan

    try:
        meteor = meteor_score(
            [reference.split()],
            generated.split()
        )
    except Exception:
        meteor = np.nan

    similarity_scores.append([
        bert, rouge1, rouge2, rougeL, meteor
    ])


similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]

results_df[similarity_columns] = pd.DataFrame(
    similarity_scores,
    index=results_df.index
)

display(results_df[["User Query"] + similarity_columns])

Similarity metrics:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.913480,0.465116,0.190476,0.348837,0.446775
1,How does AI differ from traditional programming?,0.855624,0.185185,0.038462,0.074074,0.121951
2,What is Machine Learning?,0.833536,0.051282,0.000000,0.051282,0.018450
3,What is Deep Learning?,0.836621,0.000000,0.000000,0.000000,0.000000
4,"What is the difference between AI, ML, and Dee...",0.861886,0.232558,0.023810,0.162791,0.087977
5,What is training data?,0.842480,0.162162,0.000000,0.108108,0.059289
6,What is a Machine Learning model?,0.841404,0.000000,0.000000,0.000000,0.000000
7,What is supervised learning?,0.848262,0.176471,0.000000,0.117647,0.066372
8,What is unsupervised learning?,0.850146,0.242424,0.064516,0.181818,0.145449
9,What is reinforcement learning?,0.838834,0.068966,0.000000,0.068966,0.027624


# STAGE 7 - RAG EVALUATION METRICS

In [ ]:
from bert_score import BERTScorer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import nltk
import numpy as np
import pandas as pd

# Download once if needed
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")


# ============================================================
# INITIALIZE METRICS
# ============================================================

bert_scorer = BERTScorer(
    lang="en",
    rescale_with_baseline=False
)

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)


# ============================================================
# STORAGE
# ============================================================

similarity_results = []


# ============================================================
# EVALUATE
# ============================================================

for _, row in results_df.iterrows():

    reference = str(row["Teacher Response"])
    generated = str(row["GeneratedAnswer"])

    if not generated.strip():
        similarity_results.append({
            "BERTScore_P": np.nan,
            "BERTScore_R": np.nan,
            "BERTScore_F1": np.nan,
            "ROUGE1": np.nan,
            "ROUGE2": np.nan,
            "ROUGEL": np.nan,
            "METEOR": np.nan
        })
        continue


    # ========================================================
    # BERTSCORE
    # ========================================================

    P, R, F1 = bert_scorer.score(
        [generated],
        [reference]
    )

    bert_p = float(P[0])
    bert_r = float(R[0])
    bert_f1 = float(F1[0])


    # ========================================================
    # ROUGE
    # ========================================================

    rouge_scores = rouge.score(
        reference,
        generated
    )

    rouge1 = rouge_scores["rouge1"].fmeasure
    rouge2 = rouge_scores["rouge2"].fmeasure
    rougeL = rouge_scores["rougeL"].fmeasure


    # ========================================================
    # METEOR
    # ========================================================

    meteor = meteor_score(
        [reference.split()],
        generated.split()
    )


    similarity_results.append({
        "BERTScore_P": bert_p,
        "BERTScore_R": bert_r,
        "BERTScore_F1": bert_f1,

        "ROUGE1": rouge1,
        "ROUGE2": rouge2,
        "ROUGEL": rougeL,

        "METEOR": meteor
    })


# ============================================================
# ADD TO RESULTS
# ============================================================

similarity_df = pd.DataFrame(similarity_results)

results_df = pd.concat(
    [
        results_df.reset_index(drop=True),
        similarity_df.reset_index(drop=True)
    ],
    axis=1
)


display(
    results_df[
        [
            "User Query",
            "BERTScore_F1",
            "ROUGE1",
            "ROUGE2",
            "ROUGEL",
            "METEOR"
        ]
    ]
)

In [ ]:
async def calculate_ragas(query, reference, generated):
    try:
        score = await answer_correctness_metric.ascore(
            user_input=query,
            response=generated,
            reference=reference
        )
        return float(score.value)
    except Exception as e:
        print("RAGAS error:", e)
        return np.nan


rag_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="RAG evaluation"
):
    query = row["User Query"]
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]
    context = row["RetrievedContext"]

    if not generated:
        rag_scores.append([np.nan] * 6)
        continue

    # RAGAS
    ragas_score = await calculate_ragas(
        query, reference, generated
    )

    # DEEPEVAL RAG METRICS
    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference,
        retrieval_context=context
    )

    metric_values = []

    for metric in [
        context_precision_metric,
        context_recall_metric,
        faithfulness_metric,
        rag_answer_relevancy_metric,
        rag_geval_metric
    ]:
        try:
            metric.measure(test_case)
            metric_values.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            metric_values.append(np.nan)

    # Order: RAGAS, precision, recall, faithfulness,
    # answer relevancy, GEval
    rag_scores.append([
        ragas_score,
        *metric_values
    ])


rag_columns = [
    "Ragas AnswerCorrectness",
    "Context Precision",
    "Context Recall",
    "Faithfulness",
    "RAG Answer Relevancy",
    "RAG GEval"
]

results_df[rag_columns] = pd.DataFrame(
    rag_scores,
    index=results_df.index
)

results_df["RAG Evaluation Score"] = results_df[
    rag_columns
].mean(axis=1)

display(results_df[["User Query"] + rag_columns + [
    "RAG Evaluation Score"
]])

RAG evaluation:   0%|          | 0/20 [00:00<?, ?it/s]

RAGAS error: name 'answer_correctness_metric' is not defined


Output()

Output()

ERROR:deepeval.retry.ollama:call timed out after 88.5s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt. Retrying: 1 time(s)...
INFO:deepeval.retry.ollama:Retrying in 2.180637319452815 s (attempt 1) after TimeoutError('call timed out after 88.5s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt.')


...

Output()

Output()

RAGAS error: name 'answer_correctness_metric' is not defined


Output()

Output()

# STAGE 8 - OPERATIONAL METRICS SUMMARY

In [21]:
operational_columns = [
    "RetrievalLatency",
    "GenerationLatency",
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]

operational_summary = pd.DataFrame({
    "Metric": operational_columns,
    "Average": [
        results_df[c].mean() for c in operational_columns
    ],
    "Total": [
        results_df[c].sum() for c in operational_columns
    ]
})

display(operational_summary.round(6))

,Metric,Average,Total
0,RetrievalLatency,0.398306,7.966110
1,GenerationLatency,1.590429,31.808588
2,Latency,1.991659,39.833170
3,InputTokens,472.300000,9446.000000
4,OutputTokens,71.250000,1425.000000
5,TotalTokens,543.550000,10871.000000
6,InputCost,0.000071,0.001417
7,OutputCost,0.000043,0.000855
8,TotalCost,0.000114,0.002272


# STAGE 9 - FINAL EVALUATION TABLE SUMMARY

In [22]:
# ROUND ALL METRIC COLUMNS

metric_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness",
    "AnswerQuality",
    *similarity_columns,
    *rag_columns,
    "RAG Evaluation Score"
]

results_df[metric_columns] = results_df[metric_columns].round(3)


# FINAL QUESTION-WISE TABLE

final_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer",
    *metric_columns,
    *operational_columns
]

final_df = results_df[final_columns].copy()

display(final_df)


# AVERAGE EVALUATION SUMMARY

summary_df = pd.DataFrame({
    "Metric": metric_columns,
    "Average Score": [
        results_df[c].mean() for c in metric_columns
    ]
})

summary_df["Average Score"] = (
    summary_df["Average Score"].round(3)
)

display(summary_df)


# SAVE OUTPUTS

final_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    "RAG_LLM_EVALUATION_SUMMARY.csv",
    index=False,
    encoding="utf-8-sig"
)

operational_summary.to_csv(
    "RAG_LLM_OPERATIONAL_SUMMARY.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation completed.")
print("Final results:", OUTPUT_CSV)
print("Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv")
print("Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv")

,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality,BERT,ROUGE-1,ROUGE-2,...,RAG Evaluation Score,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"AI, or artificial intelligence, is the ability...",0.900,1.000,1.000,0.967,0.936,0.541,0.386,...,0.880,1.441383,3.536660,4.990236,534,72,606,0.000080,0.000043,0.000123
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,0.911,1.000,1.000,0.970,0.896,0.327,0.196,...,0.971,0.718855,1.794741,2.514886,533,87,620,0.000080,0.000052,0.000132
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,0.921,1.000,1.000,0.974,0.909,0.444,0.278,...,0.896,0.257677,1.422330,1.681944,394,58,452,0.000059,0.000035,0.000094
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,0.912,1.000,0.909,0.940,0.941,0.606,0.469,...,0.916,0.241056,1.090079,1.332338,447,47,494,0.000067,0.000028,0.000095
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,"The difference among AI, ML, and Deep Learning...",1.000,1.000,0.903,0.968,0.913,0.598,0.353,...,0.975,0.382961,1.331282,1.716430,455,64,519,0.000068,0.000038,0.000107
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,0.803,1.000,0.980,0.928,0.896,0.391,0.200,...,0.876,0.300669,1.430209,1.733045,489,72,561,0.000073,0.000043,0.000117
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is the mathematical s...,0.904,1.000,1.000,0.968,0.900,0.450,0.154,...,0.925,0.306214,1.535233,1.843204,461,67,528,0.000069,0.000040,0.000109
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,0.993,1.000,1.000,0.998,0.913,0.458,0.173,...,0.925,0.303528,1.226223,1.532739,406,64,470,0.000061,0.000038,0.000099
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,0.980,1.000,0.993,0.991,0.929,0.439,0.175,...,0.903,0.306153,1.324377,1.632719,408,70,478,0.000061,0.000042,0.000103
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,0.908,1.000,1.000,0.969,0.926,0.453,0.219,...,0.825,0.305573,1.634641,1.942518,531,66,597,0.000080,0.000040,0.000119


,Metric,Average Score
0,AnswerCorrectness,0.907
1,AnswerRelevancy,0.983
2,Helpfulness,0.974
3,AnswerQuality,0.955
4,BERT,0.912
5,ROUGE-1,0.430
6,ROUGE-2,0.239
7,ROUGE-L,0.368
8,METEOR,0.501
9,Ragas AnswerCorrectness,0.691


Evaluation completed.
Final results: RAG_LLM_FINAL_EVALUATION.csv
Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv
Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv
